# Chapter 13 — The Galactic bar: orbits, resonances and the Hercules stream

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vasilybelokurov/galactic-dynamics-tutorial/blob/main/notebooks/13_galactic_bar.ipynb)

**Goal.** The inner Milky Way is a rotating bar about 5 kpc long. So far every potential in this tutorial has been
static and, from Chapter 4 on, axisymmetric. Here we add the bar to a Milky Way model and see what a rotating,
non-axisymmetric potential does to orbits: energy and angular momentum are no longer conserved, but one combination
of them is; stars at **resonances** with the bar behave quite differently from the rest. Then we test the classic
application near the Sun: the **Hercules stream**, a group of stars in the local velocity distribution that is
thought to be the bar's imprint.

**You will learn**
1. which bar models AGAMA offers, and how to add a realistic bar to a Milky Way potential and make it rotate;
2. to integrate orbits in the **rotating frame** of the bar and in the inertial frame, and to check the conserved
   **Jacobi integral** $E_J = E - \Omega_b L_z$;
3. the **effective potential**, the **Lagrange points** and the **corotation** radius;
4. the **resonances** $m(\Omega_\phi - \Omega_b) = l\,\Omega_R$, where they lie in the Milky Way, and the orbit families they create;
5. to predict the local velocity distribution in a barred galaxy by **backward integration** (Dehnen 2000), and to
   compare it with the Gaia data of Chapter 11 — and see why Hercules alone does not fix the bar's pattern speed.

**Data.** The file `solar_neighbourhood` of the tutorial's Zenodo record (https://doi.org/10.5281/zenodo.23646482),
introduced in Chapter 11.

**Units.** kpc, km/s, $M_\odot$. The AGAMA time unit is then 1 kpc/(km/s) = 0.978 Gyr, and frequencies in km/s/kpc
are inverse time units, so a pattern speed in km/s/kpc can be passed to AGAMA as it is.

**Conventions.** As in Chapter 11: the Sun is at $x = -R_0$, $y = 0$, and the Galaxy rotates **clockwise** seen from
the North Galactic Pole, so disk stars have $L_z = x v_y - y v_x < 0$. We quote rotation speeds $v_\phi = -L_z/R$ and the
pattern speed $\Omega_b$ as positive numbers; in AGAMA calls a clockwise rotation is a **negative** angular velocity.
(Many papers mirror the frame so that the Galaxy rotates anticlockwise; then all these signs flip.)

In [ ]:
import sys, importlib.util
if importlib.util.find_spec('agama') is None:      # install AGAMA on Colab (see Chapter 0)
    v = f'cp{sys.version_info.major}{sys.version_info.minor}'
    !pip install -q "https://github.com/vasilybelokurov/galactic-dynamics-tutorial/releases/download/agama-wheels/agama-1.0-{v}-{v}-linux_x86_64.whl"
import agama

agama.setUnits(mass=1, length=1, velocity=1)       # Msun, kpc, km/s -- before creating any AGAMA object

import os, hashlib, urllib.request, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from scipy.optimize import brentq
from scipy.ndimage import gaussian_filter, gaussian_filter1d
from scipy.signal import argrelmin

# Zenodo record 10.5281/zenodo.23646482 (version 2); as in Chapter 11
ZENODO = 'https://zenodo.org/records/23646482/files/{}?download=1'
MD5 = {'solar_neighbourhood': 'f6119c98ae50ded14fab91d1d4c071fd', 'halo_gse': '1c0533ce89d6d95ff67406dcf2a95f31',
       'disc_giants': '12f6a9ca0ac3d8ecdcd71988267b8ea7', 'rotation_curves': 'a41c95e4ecc31b3ab3bde99880736c6e',
       'dsph_gaia': '0b135e7f08816209b379f892196307b2'}


def load_data(name, cache=Path('data')):
    """Download data/<name>.parquet from Zenodo once, check its MD5, and return it as a pandas DataFrame."""
    path = cache / f'{name}.parquet'
    if not path.exists():
        cache.mkdir(exist_ok=True)
        print(f'downloading {path.name} from Zenodo ...')
        urllib.request.urlretrieve(ZENODO.format(path.name), path)
    md5 = hashlib.md5(path.read_bytes()).hexdigest()
    assert md5 == MD5[name], f'{path} is corrupt (md5 {md5}); delete it and rerun'
    return pd.read_parquet(path)


R0 = 8.122                    # kpc, Sun-Galactic centre distance (Chapter 11)
GYR = 0.9778                  # Gyr per AGAMA time unit

## 13.1 Bars in AGAMA

A bar is a triaxial density figure that rotates rigidly with a **pattern speed** $\Omega_b$. AGAMA offers several ways
to describe one:

| model | AGAMA | comment |
|---|---|---|
| Ferrers ellipsoid | `type='Ferrers', mass=, scaleRadius=, axisRatioY=, axisRatioZ=` | homogeneous-like ellipsoid, $\rho \propto (1 - m^2)^2$; analytic forces; the classic bar of orbit studies (Pfenniger 1984) |
| triaxial Dehnen model | `type='Dehnen', gamma=, axisRatioY=, axisRatioZ=` | cuspy triaxial bulge/bar; better wrapped in a `Multipole` expansion |
| any density | `type='CylSpline', density=...` or `type='Multipole', density=...` | the expansion turns any density — an analytic function, or an $N$-body snapshot — into a potential |
| a ready Milky Way | `agama/py/example_mw_bar_potential.py`, `example_mw_potential_hunter24.py` | the bar of Portail et al. (2017), fitted analytically by Sormani et al. (2022), inside a full Milky Way model (Hunter et al. 2024) |

Any of them is made to rotate by the **`rotation` modifier**, `agama.Potential(potential=pot, rotation=[[t1, angle1], [t2, angle2], ...])`,
which turns the potential about the $z$ axis by an angle that is interpolated in time; two points with angles
$\phi_0$ and $\phi_0 + \Omega_b \cdot 1$ at $t = 0$ and $t = 1$ make a constant pattern speed. Similarly the **`scale` modifier**
multiplies the potential by a time-dependent amplitude, which we will use to grow the bar slowly.

We use the realistic model: the Milky Way potential of Hunter et al. (2024), whose bar is the made-to-measure model
of the bulge and bar by Portail et al. (2017) as fitted by Sormani et al. (2022) — an X-shaped inner bar and two
thinner "long bar" components — together with stellar and gas disks, a nuclear disk and cluster, and an Einasto dark
halo. The density of the bar is defined in a file shipped with AGAMA; the other components are written out below as
in AGAMA's `example_mw_potential_hunter24.py`. All the flattened and triaxial components go into one `CylSpline`
expansion in $(R, z, \phi)$ with azimuthal harmonics up to $m_{\max} = 8$; the spherical ones into a `Multipole`.

In [ ]:
from agama.py.example_mw_bar_potential import makeBarDensity    # Sormani et al. (2022) fit to the Portail et al. (2017) bar

with np.errstate(all='ignore'):        # the analytic bar density overflows harmlessly far from the bar
    dens_bar = makeBarDensity()        # an agama.Density, bar major axis along x, not rotating
    disks = [dict(type='Disk', surfaceDensity=1.332e9, scaleRadius=2.0, scaleHeight=0.3, innerCutoffRadius=2.7),   # thin
             dict(type='Disk', surfaceDensity=8.97e8, scaleRadius=2.8, scaleHeight=0.9, innerCutoffRadius=2.7),    # thick
             dict(type='Disk', surfaceDensity=5.81e7, scaleRadius=7, scaleHeight=-0.085, innerCutoffRadius=4),     # HI
             dict(type='Disk', surfaceDensity=2.68e9, scaleRadius=1.5, scaleHeight=-0.045, innerCutoffRadius=12)]  # H2
    spheroids = [dict(type='Spheroid', densitynorm=2.774e11, gamma=0, beta=0, alpha=1,                 # Einasto dark halo
                      outerCutoffRadius=8.682e-6, cutoffStrength=0.1704),
                 dict(type='Plummer', mass=4.1e6, scaleRadius=1e-3),                                      # Sgr A*
                 dict(type='Spheroid', mass=6.1e7, gamma=0.71, beta=4, alpha=1, axisRatioZ=0.73,          # nuclear star cluster
                      scaleRadius=0.0059, outerCutoffRadius=0.1),
                 dict(type='Spheroid', densityNorm=2.00583e12, gamma=0, beta=0, alpha=1, axisRatioZ=0.37,  # nuclear stellar disk
                      outerCutoffRadius=0.00506, cutoffStrength=0.72),
                 dict(type='Spheroid', densityNorm=1.53e12, gamma=0, beta=0, alpha=1, axisRatioZ=0.37,
                      outerCutoffRadius=0.0246, cutoffStrength=0.79)]
    pot_sph = agama.Potential(type='Multipole', density=agama.Density(*spheroids), lmax=12, gridSizeR=36, rmin=1e-4, rmax=1000)
    cyl = dict(type='CylSpline', density=agama.Density(dens_bar, *disks),
               gridSizeR=30, gridSizez=32, Rmin=0.1, Rmax=200, zmin=0.05, zmax=200)
    t0 = time.time()
    pot_bar = agama.Potential(pot_sph, agama.Potential(mmax=8, **cyl))     # barred Milky Way (bar along x, static)
    pot_axi = agama.Potential(pot_sph, agama.Potential(mmax=0, **cyl))     # the same, keeping only the m = 0 harmonic
print(f'potentials built in {time.time() - t0:.1f} s; bar density mass {dens_bar.totalMass():.2e} Msun')

`pot_axi` keeps only the $m = 0$ (axisymmetric) term of the same expansion. It should be the azimuthal average of
`pot_bar`; we check this, because we will use `pot_axi` for everything that needs an axisymmetric potential: the
circular velocity, epicyclic frequencies, actions.

In [ ]:
phi = np.linspace(0, 2 * np.pi, 73)[:-1]
for R in (1, 3, 5, 8, 12):
    ring = np.column_stack([R * np.cos(phi), R * np.sin(phi), 0 * phi])
    dev = pot_bar.potential(ring).mean() / pot_axi.potential([R, 0, 0]) - 1
    print(f'R = {R:2d} kpc: azimuthal mean of the barred potential / axisymmetric - 1 = {dev:.1e}')
    assert abs(dev) < 1e-4

For comparison we also make a Ferrers bar of the same mass and length. Figure 13.1 shows the surface density of both,
seen from the North Galactic Pole and rotated to the bar's orientation in the Milky Way: the near end of the bar
points towards positive Galactic longitude, at an angle of about $28°$ to the Sun–Galactic centre line (Wegg, Gerhard &
Portail 2015). A rotation of the bar from the $x$ axis by an angle $\alpha$ anticlockwise is `rotation=α`; ours is
$\alpha = -28°$.

In [ ]:
bar_angle = np.radians(-28.0)        # orientation of the bar major axis today, measured anticlockwise from +x
ferrers = agama.Density(type='Ferrers', mass=dens_bar.totalMass(), scaleRadius=5.0, axisRatioY=0.35, axisRatioZ=0.2)

g = np.linspace(-10, 10, 201)
X, Y = np.meshgrid(g, g, indexing='ij')
XY = np.column_stack([X.ravel(), Y.ravel()])
rot = np.array([[np.cos(bar_angle), np.sin(bar_angle)], [-np.sin(bar_angle), np.cos(bar_angle)]])   # sky -> bar frame
XYbar = XY @ rot.T


def surface_density(dens, xy, zmax=5.0, nz=121):
    """Face-on surface density: integrate dens.density along z on a grid that is dense near the plane.
    (AGAMA's dens.projectedDensity(xy) does the same, but prints a progress line for large grids.)"""
    z = 0.02 * np.sinh(np.linspace(-1, 1, nz) * np.arcsinh(zmax / 0.02))      # spacing 0.02 kpc near z = 0
    pts = np.column_stack([np.repeat(xy, nz, axis=0), np.tile(z, len(xy))])
    return np.trapezoid(dens.density(pts).reshape(len(xy), nz), z, axis=1)


with np.errstate(all='ignore'):
    Sig_bar = surface_density(agama.Density(dens_bar, *disks), XYbar).reshape(X.shape)   # face-on surface density
    Sig_fer = surface_density(ferrers, XYbar).reshape(X.shape)
    Sig_bar_only = surface_density(dens_bar, XY).reshape(X.shape)                       # the bar alone, along x

fig, ax = plt.subplots(1, 3, figsize=(17, 5))
for a, S, title in [(ax[0], Sig_fer, 'Ferrers bar'), (ax[1], Sig_bar, 'Portail/Sormani bar + disks')]:
    a.contourf(X, Y, np.log10(np.maximum(S, 1e5)), levels=np.linspace(6.5, 10.5, 17), cmap='Greys')
    a.plot(-R0, 0, 'o', color='gold', mec='k', ms=10); a.text(-R0 + 0.3, 0.5, 'Sun')
    a.plot([-R0, 0], [0, 0], 'k:', lw=1)
    a.set_aspect('equal'); a.set_xlabel('$x$ [kpc]'); a.set_ylabel('$y$ [kpc]'); a.set_title(title)
ax[1].annotate('rotation', xy=(5.5, -5.5), xytext=(7.5, -1.5), arrowprops=dict(arrowstyle='->', connectionstyle='arc3,rad=-0.4'))
# circular velocity curves of the axisymmetric part, and the strength of the bar's tangential force
R = np.linspace(0.2, 15, 150)
xyz = lambda R: np.column_stack([R, 0 * R, 0 * R])
vc = np.sqrt(-R * pot_axi.force(xyz(R))[:, 0])
Q_T = []                    # Q_T(R) = max over phi of |F_phi| / |<F_R>|, the usual measure of bar strength
for Ri in R:
    ring = np.column_stack([Ri * np.cos(phi), Ri * np.sin(phi), 0 * phi])
    F = pot_bar.force(ring)
    F_phi = -np.sin(phi) * F[:, 0] + np.cos(phi) * F[:, 1]
    F_R = np.cos(phi) * F[:, 0] + np.sin(phi) * F[:, 1]
    Q_T.append(np.max(np.abs(F_phi)) / np.abs(F_R.mean()))
Q_T = np.array(Q_T)
ax[2].plot(R, vc, 'k', label='$v_c$ (axisymmetric part) [km/s]')
a2 = ax[2].twinx(); a2.plot(R, Q_T, 'C3'); a2.set_ylabel('$Q_T$', color='C3')
ax[2].set_xlabel('$R$ [kpc]'); ax[2].set_ylabel('$v_c$ [km/s]'); ax[2].set_ylim(0, 280)
plt.tight_layout(); plt.show()
print(f'v_c(R0) = {np.interp(R0, R, vc):.1f} km/s; bar strength peaks at Q_T = {Q_T.max():.2f} at R = {R[np.argmax(Q_T)]:.1f} kpc, '
      f'Q_T(R0) = {np.interp(R0, R, Q_T):.3f}')
assert 0.15 < Q_T.max() < 0.6 and np.interp(R0, R, Q_T) < 0.05

**Figure 13.1.** Left and middle: face-on surface density (log scale) of a Ferrers bar and of the Portail/Sormani bar
with the stellar and gas disks, rotated to the present orientation of the Milky Way bar; the Sun is at $x = -8.1$ kpc and
the Galaxy rotates clockwise. Right: circular velocity of the axisymmetric part of the model (black) and the bar
strength $Q_T(R)$, the maximum tangential force on a circle relative to the mean radial force (red).

The Portail/Sormani bar (Figure 13.1, middle) is boxy in its centre, where the X-shaped bulge lives, and continues as a
thin "long bar" to about 5 kpc. Its tangential force reaches 31% of the radial force at 2.6 kpc, but only a
per cent at the Sun (right). This weak force at the Sun is what makes the bar's effects there subtle, and resonances
— where small forces add up coherently over many orbits — important.

## 13.2 Orbits in a rotating bar

There are two equivalent ways to integrate an orbit in a bar that rotates with a constant $\Omega_b$:

1. make the potential rotate with the `rotation` modifier and integrate in the **inertial** frame;
2. keep the potential static and integrate in a frame that **rotates with the bar**, `agama.orbit(..., Omega=Ω, verbose=False)`.

In the second case AGAMA returns positions in the rotating frame, while the velocities are inertial velocities
expressed along the rotating axes (they are what an inertial observer would measure, momentarily aligned with the
bar). We use the pattern speed of the Hunter et al. model, $\Omega_b = 37.5$ km/s/kpc, and integrate one disk orbit both
ways for 2 time units (2 Gyr).

In [ ]:
Omega_b = 37.5                         # km/s/kpc; clockwise, so -Omega_b in AGAMA
pot_rot = agama.Potential(potential=pot_bar, rotation=[[0, 0], [1, -Omega_b]])    # bar angle = -Omega_b * t

ic = np.array([-6.0, 0.5, 0.05, 30., 190., 5.])      # a disk star at R = 6 kpc, inertial position and velocity
T, n = 2.0, 2001
t, tr_in = agama.orbit(potential=pot_rot, ic=ic, time=T, trajsize=n, verbose=False)                 # 1. inertial frame
_, tr_rf = agama.orbit(potential=pot_bar, ic=ic, time=T, trajsize=n, Omega=-Omega_b, verbose=False)  # 2. rotating frame


def to_bar_frame(xy, t):
    """Rotate inertial (x, y) positions at times t into the frame of a bar whose angle is -Omega_b * t."""
    a = Omega_b * t                    # rotate by +Omega_b t to undo the bar's clockwise turn
    return np.column_stack([np.cos(a) * xy[:, 0] - np.sin(a) * xy[:, 1], np.sin(a) * xy[:, 0] + np.cos(a) * xy[:, 1]])


diff = np.max(np.linalg.norm(to_bar_frame(tr_in[:, :2], t) - tr_rf[:, :2], axis=1))
print(f'the two integrations agree to {diff:.1e} kpc')
assert diff < 1e-4

In a rotating potential the energy $E$ is not conserved — the potential at a fixed point changes with time — and
neither is $L_z$, because the bar exerts a torque. But in the frame of the bar the potential is static, and the energy
in that frame is conserved. This is the **Jacobi integral**

$$ E_J = E - \boldsymbol\Omega_b \cdot \mathbf L = \tfrac12 |\mathbf v|^2 + \Phi(\mathbf x) - \Omega_b L_z, $$

where $E = \frac12|\mathbf v|^2 + \Phi(\mathbf x)$ is the energy per unit mass, $\mathbf x$ the position and $\mathbf v$ the inertial velocity,
$\Phi(\mathbf x)$ the potential of the bar at its current orientation, $\boldsymbol\Omega_b$ the bar's angular-velocity vector (along $z$) and
$\mathbf L = \mathbf x \times \mathbf v$ the angular momentum; $\Omega_b$ and $L_z$ are their signed $z$ components (here both negative). Equivalently,
with the velocity $\mathbf v_{\rm rot} = \mathbf v - \boldsymbol\Omega_b \times \mathbf x$ measured in the rotating frame,

$$ E_J = \tfrac12 |\mathbf v_{\rm rot}|^2 + \Phi_{\rm eff}(\mathbf x), \qquad \Phi_{\rm eff} = \Phi - \tfrac12 \Omega_b^2 R^2 , $$

the kinetic energy in the rotating frame plus an **effective potential** that includes the centrifugal term ($R$ is the
cylindrical radius, the distance from the rotation axis). We
compute $E_J$ both ways along the orbit (Figure 13.2, right).

In [ ]:
x, v = tr_rf[:, :3], tr_rf[:, 3:]                     # rotating-frame positions, inertial velocities
E = 0.5 * np.sum(v**2, axis=1) + pot_bar.potential(x)
Lz = x[:, 0] * v[:, 1] - x[:, 1] * v[:, 0]
EJ = E - (-Omega_b) * Lz                               # signed Omega = -Omega_b
Om_vec = np.array([0, 0, -Omega_b])
v_rot = v - np.cross(Om_vec, x)                        # velocity in the rotating frame
EJ2 = 0.5 * np.sum(v_rot**2, axis=1) + pot_bar.potential(x) - 0.5 * Omega_b**2 * (x[:, 0]**2 + x[:, 1]**2)
rel = lambda q: np.ptp(q) / abs(q.mean())
print(f'relative range along the orbit: E {rel(E):.3f}, L_z {rel(Lz):.3f}, E_J {rel(EJ):.1e}; '
      f'the two forms of E_J differ by {np.max(np.abs(EJ2 / EJ - 1)):.1e}')
assert rel(E) > 0.01 and rel(EJ) < 1e-5 and np.max(np.abs(EJ2 / EJ - 1)) < 1e-6

fig, ax = plt.subplots(1, 3, figsize=(16, 5))
ax[0].plot(tr_in[:, 0], tr_in[:, 1], lw=0.5); ax[0].set_title('inertial frame')
ax[1].plot(tr_rf[:, 0], tr_rf[:, 1], lw=0.5, color='C1'); ax[1].set_title('frame rotating with the bar')
for a in ax[:2]:
    a.contour(g, g, np.log10(np.maximum(Sig_bar_only, 1e5)).T,
              levels=[8, 9, 10], colors='k', linewidths=0.6)
    a.set_aspect('equal'); a.set_xlim(-10, 10); a.set_ylim(-10, 10); a.set_xlabel('$x$ [kpc]'); a.set_ylabel('$y$ [kpc]')
ax[0].text(0, -9, 'bar contours at $t = 0$', ha='center')
ax[2].plot(t * GYR, E / abs(E[0]) - E[0] / abs(E[0]), label='$E$')
ax[2].plot(t * GYR, Lz / abs(Lz[0]) - Lz[0] / abs(Lz[0]), label='$L_z$')
ax[2].plot(t * GYR, EJ / abs(EJ[0]) - EJ[0] / abs(EJ[0]), 'k', lw=2, label='$E_J$')
ax[2].set_xlabel('$t$ [Gyr]'); ax[2].set_ylabel('relative change'); ax[2].legend()
plt.tight_layout(); plt.show()

**Figure 13.2.** One orbit in the barred Milky Way, integrated for 2 Gyr. Left: in the inertial frame. Middle: in the frame
rotating with the bar, with contours of the bar's surface density. Right: relative changes of the energy $E$, angular
momentum $L_z$ and Jacobi integral $E_J$ along the orbit.

The orbit, a rosette in the inertial frame (Figure 13.2, left), looks quite different in the bar frame (middle). At
$R \approx 4$–6 kpc the star goes round the Galaxy only a little faster than the bar, so in the bar frame it advances slowly,
looping once per radial oscillation, and the bar has time to push it. Its energy changes by about 10% and its
angular momentum by 40% — the bar exchanges both with the star — but always so that $E_J$ stays constant to $10^{-7}$
(right): $\Delta E = \Omega_b\, \Delta L_z$. A rotating bar moves stars along lines of slope $\Omega_b$ in the $(L_z, E)$ plane.

## 13.3 The effective potential and the Lagrange points

$\Phi_{\rm eff}$ (Figure 13.3) has five stationary points where a star can stay at rest in the bar frame, the
**Lagrange points**, as in the restricted three-body problem: $L_3$ at the centre, two saddles $L_1$, $L_2$ on the bar's
major axis and two maxima $L_4$, $L_5$ on its minor axis. Since $E_J \geq \Phi_{\rm eff}$, a star with $E_J$ below the value at
$L_1$ cannot cross the contour of $\Phi_{\rm eff} = E_J$ (a **zero-velocity curve**): inside corotation it is confined to
the bar region, outside it to the outer disk.

In [ ]:
Peff = lambda x, y: pot_bar.potential(np.column_stack([x, y, 0 * x])) - 0.5 * Omega_b**2 * (x**2 + y**2)
d_dx = lambda x: (Peff(np.array([x + 1e-4]), np.array([0.])) - Peff(np.array([x - 1e-4]), np.array([0.])))[0] / 2e-4
d_dy = lambda y: (Peff(np.array([0.]), np.array([y + 1e-4])) - Peff(np.array([0.]), np.array([y - 1e-4])))[0] / 2e-4
L1 = brentq(d_dx, 3, 9)                  # saddle on the major axis
L4 = brentq(d_dy, 3, 9)                  # maximum on the minor axis


def frequencies(R):
    """Circular frequency Omega and epicyclic frequency kappa [km/s/kpc] of the axisymmetric part (Chapter 5)."""
    h = 1e-3
    F = lambda r: -pot_axi.force(xyz(np.atleast_1d(r)))[:, 0]          # -dPhi/dR
    Om = np.sqrt(F(R) / R)
    kappa = np.sqrt((F(R + h) - F(R - h)) / (2 * h) + 3 * F(R) / R)   # kappa^2 = d2Phi/dR2 + 3/R dPhi/dR
    return Om, kappa


R_fine = np.linspace(0.3, 16, 4000)
Om_R, kap_R = frequencies(R_fine)
R_CR = R_fine[np.argmin(np.abs(Om_R - Omega_b))]
print(f'Lagrange points: L1 at {L1:.2f} kpc (major axis), L4 at {L4:.2f} kpc (minor axis); '
      f'corotation of the axisymmetric part Omega(R) = Omega_b at {R_CR:.2f} kpc')
assert abs(L1 / R_CR - 1) < 0.08 and abs(L4 / R_CR - 1) < 0.05

gl = np.linspace(-9, 9, 181)
GX, GY = np.meshgrid(gl, gl, indexing='ij')
PE = Peff(GX.ravel(), GY.ravel()).reshape(GX.shape)
fig, ax = plt.subplots(1, 2, figsize=(12, 5.5))
P1, P4 = Peff(np.array([L1]), np.array([0.]))[0], Peff(np.array([0.]), np.array([L4]))[0]
ax[0].contour(GX, GY, PE / 1e5, levels=np.sort(np.concatenate([np.linspace(PE.min(), P1, 12)[:-1], np.linspace(P1, P4, 6)])) / 1e5,
              colors='0.4', linewidths=0.7, linestyles='solid')
ax[0].contour(GX, GY, PE, levels=[P1], colors='C3', linewidths=2, linestyles='solid')
for name, (px, py) in {'L1': (L1, 0), 'L2': (-L1, 0), 'L4': (0, L4), 'L5': (0, -L4), 'L3': (0, 0)}.items():
    ax[0].plot(px, py, 'k+', ms=12, mew=2); ax[0].text(px + 0.3, py + 0.3, name)
ax[0].add_patch(plt.Circle((0, 0), R_CR, fill=False, ls='--', color='C0'))
ax[0].set_aspect('equal'); ax[0].set_xlabel('$x$ (bar major axis) [kpc]'); ax[0].set_ylabel('$y$ [kpc]')
ax[1].plot(gl, Peff(gl, 0 * gl) / 1e5, label='along the major axis')
ax[1].plot(gl, Peff(0 * gl, gl) / 1e5, label='along the minor axis')
ax[1].set_xlabel('distance from the centre [kpc]'); ax[1].set_ylabel('$\\Phi_{\\rm eff}$ [$10^5$ (km/s)$^2$]'); ax[1].legend()
plt.tight_layout(); plt.show()

**Figure 13.3.** Left: contours of the effective potential $\Phi_{\rm eff} = \Phi - \frac12\Omega_b^2R^2$ in the bar frame for
$\Omega_b = 37.5$ km/s/kpc (bar along $x$), the Lagrange points, the corotation circle of the axisymmetric part (dashed) and
the contour through $L_1$ and $L_2$ (red). Right: $\Phi_{\rm eff}$ along the bar's major and minor axes.

The Lagrange points lie close to the corotation radius $R_{\rm CR} = 6.1$ kpc, where a circular orbit in the axisymmetric
part has $\Omega(R) = \Omega_b$; $L_1$/$L_2$ a little outside it and $L_4$/$L_5$ a little inside (Figure 13.3). The bar's half-length
of about 5 kpc is just inside corotation, as for a bar that is "fast" in the usual sense ($R_{\rm CR}/R_{\rm bar} \approx 1.2$, with
$R_{\rm bar}$ the bar's half-length; Portail
et al. 2017). Bars cannot extend beyond corotation: orbits there are elongated perpendicular to the bar and cannot
support it (Contopoulos 1980; Contopoulos 1989).

## 13.4 Resonances

A star on a near-circular orbit at radius $R$ goes round the Galaxy with angular frequency $\Omega(R)$ and oscillates
radially with the epicyclic frequency $\kappa(R)$ (Chapter 5). Relative to the bar it goes round with $\Omega - \Omega_b$, and it
meets the bar's $m$-fold pattern at frequency $m(\Omega - \Omega_b)$. When this is a multiple of $\kappa$ the bar's pushes always
arrive at the same phase of the radial oscillation, and add up: a **resonance**,

$$ m\,(\Omega_\phi - \Omega_b) = l\,\Omega_R , $$

where $m$ is the azimuthal wavenumber of the perturbation (the number of times its pattern repeats around a circle; $m = 2$
for the main, bisymmetric part of a bar), $l$ is an integer ($0, \pm 1, \pm 2, \ldots$), $\Omega_b$ the pattern speed, and $\Omega_\phi$ and
$\Omega_R$ the azimuthal and radial frequencies of the star's orbit (Chapter 9); for near-circular orbits $\Omega_\phi \to \Omega$ and
$\Omega_R \to \kappa$. Frequencies are in km/s/kpc, and here $\Omega_\phi$ is counted positive for prograde motion. For the bar, $m = 2$:
**corotation** (CR, $l = 0$), the **inner Lindblad resonance** (ILR, $l = +1$, $\Omega_b = \Omega - \kappa/2$) and the **outer Lindblad
resonance** (OLR, $l = -1$, $\Omega_b = \Omega + \kappa/2$); the weaker $m = 4$ term of the bar adds the inner and outer **4:1**
resonances ($l = \pm 1$, $\Omega_b = \Omega \mp \kappa/4$). Figure 13.4 shows where they lie for three
measured pattern speeds.

In [ ]:
speeds = {'Clarke & Gerhard 2022': 33.3, 'Hunter et al. 2024': 37.5, 'Sanders et al. 2019': 41.0}
fig, ax = plt.subplots(figsize=(8, 5))
for curve, lab, st in [(Om_R, '$\\Omega$', '-'), (Om_R - kap_R / 2, '$\\Omega - \\kappa/2$', '-.'),
                       (Om_R + kap_R / 2, '$\\Omega + \\kappa/2$', '--'), (Om_R + kap_R / 4, '$\\Omega + \\kappa/4$', ':')]:
    ax.plot(R_fine, curve, 'k', ls=st, label=lab)
res = {}
for (who, Ob), c in zip(speeds.items(), ['C0', 'C1', 'C3']):
    ax.axhline(Ob, color=c, lw=1.5, label=f'$\\Omega_b$ = {Ob} ({who})')
    cross = lambda f: R_fine[np.argmin(np.abs(f - Ob))]
    res[Ob] = dict(CR=cross(Om_R), OLR=cross(Om_R + kap_R / 2), OLR4=cross(Om_R + kap_R / 4))
    print(f'Omega_b = {Ob}: R_CR = {res[Ob]["CR"]:.2f}, R_4:1 = {res[Ob]["OLR4"]:.2f}, R_OLR = {res[Ob]["OLR"]:.2f} kpc')
ax.axvline(R0, color='gold', lw=3, alpha=0.6); ax.text(R0 + 0.1, 92, 'Sun')
ax.set_ylim(0, 100); ax.set_xlim(0, 16); ax.set_xlabel('$R$ [kpc]'); ax.set_ylabel('frequency [km/s/kpc]'); ax.legend(fontsize=8)
plt.show()
assert 6.5 < res[33.3]['CR'] < 7.5 and 10.7 < res[33.3]['OLR'] < 12.4       # the ranges quoted by Clarke & Gerhard (2022)

**Figure 13.4.** Circular frequency $\Omega$, $\Omega \pm \kappa/2$ and $\Omega + \kappa/4$ of the axisymmetric part of the model against radius,
with three measured pattern speeds (horizontal lines); a resonance sits where a line crosses a curve. The yellow band is the Sun.

For the slow bar of Clarke & Gerhard (2022), corotation is at 6.9 kpc and the OLR at 11.4 kpc — inside the ranges they
quote for their own rotation curve, 6.5–7.5 and 10.7–12.4 kpc. For the faster bar of Sanders, Smith & Evans (2019),
the 4:1 resonance falls almost exactly at the Sun (Figure 13.4). Measurements of $\Omega_b$ in the last decade range
from about 31 to 41 km/s/kpc; the ILR is inside 1.5 kpc for all of them.

What do resonant orbits look like? A **closed** orbit in the bar frame is a resonant orbit: it returns to the same
place relative to the bar. We find closed orbits by **shooting**: launch a star from the bar's minor axis, moving
parallel to the major axis, and adjust its speed until it crosses the minor axis again at a right angle — by symmetry
the orbit then closes. Figure 13.5 shows four kinds:

- **x1** orbits inside the bar, elongated along it: the backbone of the bar;
- orbits trapped at **corotation**, librating around $L_4$ ("banana" orbits);
- the outer **4:1** resonance, square-ish orbits;
- near the **OLR**, the closed orbits are elongated *perpendicular* to the bar inside $R_{\rm OLR}$ and *parallel* to it outside.

In [ ]:
Om_signed = -Omega_b


def ic_rotating(x, y, ux, uy):
    """AGAMA initial condition for a star at (x, y) moving with velocity (ux, uy) in the bar frame."""
    return np.array([x, y, 0, ux - Om_signed * y, uy + Om_signed * x, 0])     # v = v_rot + Omega x r


def orbit_bar(ic, T, n=4000):
    return agama.orbit(potential=pot_bar, ic=ic, time=T, trajsize=n, Omega=Om_signed, dtype=float, verbose=False)[1]   # float64: see below


def crossing_velocity(u, y0):
    """Launch from (0, y0) with bar-frame velocity (u, 0); return v_y (bar frame) at the next crossing of x = 0."""
    tr = orbit_bar(ic_rotating(0, y0, u, 0), 2.0, 20000)
    vy = tr[:, 4] + Om_signed * tr[:, 0]                     # bar-frame v_y = v_y - (Omega x r)_y
    k = np.flatnonzero((np.sign(tr[1:, 0]) != np.sign(tr[:-1, 0])) & (np.arange(len(tr) - 1) > 5))
    return vy[k[0]] if len(k) else np.nan


def closed_orbit(y0, u0, du=30):
    """The launch speed closest to u0 for which the orbit through (0, y0) closes."""
    u_grid = np.linspace(u0 - du, u0 + du, 31)
    m = np.array([crossing_velocity(u, y0) for u in u_grid])
    k = np.flatnonzero(np.sign(m[1:]) != np.sign(m[:-1]))
    roots = [brentq(crossing_velocity, u_grid[i], u_grid[i + 1], args=(y0,)) for i in k]
    return roots[np.argmin(np.abs(np.array(roots) - u0))]


u_x1 = closed_orbit(1.0, 220.0)                                  # x1: launched 1 kpc from the centre on the minor axis
x1 = orbit_bar(ic_rotating(0, 1.0, u_x1, 0), 1.0)
banana = [orbit_bar(ic_rotating(0, L4 + dy, 0, 0), 6, 6000) for dy in (0.25, 0.5)]   # start at rest near L4
R41, ROLR = res[Omega_b]['OLR4'], res[Omega_b]['OLR']
vc_at = lambda R: np.sqrt(-R * pot_axi.force([R, 0, 0])[0])
four = orbit_bar(ic_rotating(0, R41, vc_at(R41) - Omega_b * R41, 0), 2)
olr = []
for y0 in (ROLR - 1.0, ROLR + 1.0):
    u0 = vc_at(y0) - Omega_b * y0                       # bar-frame speed of a circular orbit
    olr.append(orbit_bar(ic_rotating(0, y0, closed_orbit(y0, u0), 0), 2))

fig, ax = plt.subplots(1, 4, figsize=(20, 5.2))
panels = [([x1], 'x1 (inside the bar)', 4), (banana, 'corotation: around $L_4$', 9),
          ([four], f'outer 4:1 ($R$ = {R41:.1f} kpc)', 11)]
for a, (orbs, title, lim) in zip(ax[:3], panels):
    a.contour(g, g, np.log10(np.maximum(Sig_bar_only, 1e5)).T,
              levels=[8, 9, 10], colors='0.6', linewidths=0.6)
    for o in orbs:
        a.plot(o[:, 0], o[:, 1], lw=0.6)
    a.add_patch(plt.Circle((0, 0), R_CR, fill=False, ls=':', color='k'))
    a.set_aspect('equal'); a.set_xlim(-lim, lim); a.set_ylim(-lim, lim); a.set_title(title); a.set_xlabel('$x$ [kpc]')
ax[0].set_ylabel('$y$ [kpc]')
# near the OLR the closed orbits are almost circles; show their shape as R(phi) / <R> - 1 against the bar-frame azimuth
c2 = []
for o, lab in zip(olr, ['1 kpc inside the OLR', '1 kpc outside the OLR']):
    ph, Ro = np.degrees(np.arctan2(o[:, 1], o[:, 0])), np.hypot(o[:, 0], o[:, 1])
    ax[3].plot(ph, Ro / Ro.mean() - 1, '.', ms=1, label=lab)
    c2.append(np.mean((Ro / Ro.mean() - 1) * np.cos(2 * np.radians(ph))))   # > 0: elongated along the bar
ax[3].set_xlabel('azimuth from the bar major axis [deg]'); ax[3].set_ylabel('$R / \\langle R \\rangle - 1$')
ax[3].set_title(f'closed orbits near the OLR ($R_{{\\rm OLR}}$ = {ROLR:.1f} kpc)'); ax[3].legend(fontsize=8, markerscale=8)
ax[3].set_xticks([-180, -90, 0, 90, 180]); ax[3].set_box_aspect(1)
plt.tight_layout(); plt.show()
print(f'elongation cos(2 phi) coefficient: inside the OLR {c2[0]:+.4f}, outside {c2[1]:+.4f}')
assert c2[0] < 0 < c2[1]

# frequency ratios (|Omega_phi| - Omega_b) / Omega_R from AGAMA's ActionFinder in the axisymmetric part:
# 0 at corotation, -1/4 at the outer 4:1, -1/2 at the OLR, +1/2 at the ILR
af = agama.ActionFinder(pot_axi)
ratio = lambda xv: (np.abs(af(xv, frequencies=True)[1][..., 2]) - Omega_b) / af(xv, frequencies=True)[1][..., 0]
print(f'frequency ratio at the start of the corotation orbit: {ratio(banana[0][0]):+.3f};  '
      f'4:1 orbit: {ratio(four[0]):+.3f};  OLR orbits: {ratio(olr[0][0]):+.3f}, {ratio(olr[1][0]):+.3f}')
print(f'x1 orbit: length {np.ptp(x1[:, 0]):.1f} kpc along the bar, width {np.ptp(x1[:, 1]):.1f} kpc')
assert np.ptp(x1[:, 0]) > 2 * np.ptp(x1[:, 1])                                  # x1 is elongated along the bar
assert abs(ratio(banana[0][0])) < 0.1 and abs(ratio(four[0]) + 0.25) < 0.05

**Figure 13.5.** Orbits in the frame of the bar ($\Omega_b = 37.5$ km/s/kpc; grey: bar density contours; dotted: corotation circle).
From left: a closed x1 orbit; two orbits started at rest near $L_4$; a closed orbit at the outer 4:1 resonance. Right: the
shape of the closed orbits 1 kpc inside and outside the OLR, as the fractional deviation of their radius from the mean
against azimuth measured from the bar's major axis.

The x1 orbit (Figure 13.5, left) traces the bar: a galaxy can build a self-consistent bar from such orbits. The stars
near $L_4$ never leave it: in the bar frame they drift slowly along a banana-shaped path around the maximum of
$\Phi_{\rm eff}$, held there by the Coriolis force; the one started further out makes a longer banana. These are the
orbits that Pérez-Villegas et al. (2017) proposed for the Hercules stream. The 4:1 orbit closes after four radial
oscillations per turn relative to the bar. Across the OLR, the closed orbits switch from perpendicular to parallel
to the bar (right; the deformation is only a few per cent at these radii, where the bar is weak): this switch is behind the Hercules model of Dehnen (2000), next section. The frequency ratios, computed by
AGAMA from the actions in the axisymmetric part, confirm the resonances: $0$ for corotation and $-1/4$ for 4:1. The two
closed orbits near the OLR have $-0.35$ (inside) and $-0.66$ (outside): neither is resonant, they lie on either side of $-1/2$.
Their elongation is the response of a forced oscillator — the bar drives the radial motion at frequency $2(\Omega_\phi - \Omega_b)$ —
and, as for any driven oscillator, the response changes phase by $180°$ when the driving frequency crosses the natural
frequency $\Omega_R$. That is why the orbits flip from perpendicular to parallel across the OLR.

## 13.5 The solar neighbourhood and the Hercules stream

The velocity distribution of stars near the Sun (Chapter 11, Figure 11.2) has a large secondary group, the
**Hercules stream**, at $v_\phi \approx 195$ km/s, about 35 km/s slower than the main peak, moving outwards ($v_R > 0$); a gap
separates the two at $v_\phi \approx 205$–215 km/s. Two explanations by the bar compete:

- **a fast bar** with the OLR just inside the Sun (Dehnen 2000; Kalnajs 1991): orbits on either side of the OLR are
  aligned differently with the bar (Figure 13.5, right), so the velocity distribution near the OLR splits in two. Dehnen
  found $\Omega_b = (1.85 \pm 0.15)\,\Omega_0$, i.e. about 52 km/s/kpc for our $\Omega_0 = v_c/R_0$ (the circular frequency at the Sun,
  with $v_c = 229$ km/s the circular speed and $R_0$ the Sun's Galactocentric radius);
- **a slow, long bar** with corotation at 6–7 kpc (Pérez-Villegas et al. 2017; Monari et al. 2019; Chiba, Friske &
  Schönrich 2021): Hercules stars orbit the Lagrange points $L_4$/$L_5$ and visit the Sun near their apocentres.

Since measurements of the bar's length (5 kpc; Wegg et al. 2015) need corotation outside 5 kpc, i.e. $\Omega_b \lesssim 42$,
the fast-bar model now has a problem. Let us see what each predicts.

### Backward integration

How do we predict $f(\mathbf v)$ at the Sun in a barred galaxy? Let the bar grow slowly in a disk that was in
equilibrium in the axisymmetric potential, with a distribution function $f_0(\mathbf J)$ of the actions (Chapter 10). By
Liouville's theorem $f$ — the phase-space density of stars, the distribution function — is conserved along orbits, so the value of $f$ at a point $(\mathbf x, \mathbf v)$ today is $f_0$ at the point
where the orbit started before the bar existed. Integrating the orbit **backwards** from $(\mathbf x, \mathbf v)$ through the
growth of the bar to a time when the potential was axisymmetric gives $f(\mathbf x, \mathbf v) = f_0(\mathbf J(\mathbf x_{\rm start}, \mathbf v_{\rm start}))$,
where $(\mathbf x_{\rm start}, \mathbf v_{\rm start})$ is the end point of the backward integration and $\mathbf J = (J_r, J_z, J_\phi)$ are its actions
in the axisymmetric potential
(Vauterin & Dejonghe 1997; Dehnen 2000). It needs one orbit per point of a velocity grid at the Sun — no particles,
no noise.

The bar grows with the smooth law of Dehnen (2000), $A(\xi) = \frac{3}{16}\xi^5 - \frac{5}{8}\xi^3 + \frac{15}{16}\xi + \frac12$, where
$\xi = 2(t - t_{\rm start})/T_{\rm grow} - 1$, clipped to $[-1, 1]$, runs from $-1$ to $1$ over the growth time $T_{\rm grow}$ that starts at
$t_{\rm start}$; so the amplitude $A$ rises smoothly from 0 to 1. We write the potential as the axisymmetric part plus $A(t)$ times the
non-axisymmetric part, $\Phi = \Phi_{\rm axi} + A(t)\,(\Phi_{\rm bar} - \Phi_{\rm axi})$, with $\Phi_{\rm axi}$ the $m = 0$ part and $\Phi_{\rm bar}$ the full barred potential, using the `scale` modifier, whose first factor multiplies the
potential. The bar grows over 1 time unit and then rotates steadily for 2 more; the integration covers 3 time units
(2.9 Gyr), about 13 rotation periods of the Sun.

In [ ]:
T_total, T_grow = 3.0, 1.0
tt = np.linspace(-T_total - 0.5, 0.5, 801)                         # times at which the amplitude is tabulated
xi = np.clip(2 * (tt + T_total) / T_grow - 1, -1, 1)
A = 3 / 16 * xi**5 - 5 / 8 * xi**3 + 15 / 16 * xi + 0.5             # 0 before t = -3, 1 after t = -2
pot_grow = agama.Potential(pot_axi,
                           agama.Potential(potential=pot_bar, scale=np.column_stack([tt, A, np.ones_like(tt)])),
                           agama.Potential(potential=pot_axi, scale=np.column_stack([tt, -A, np.ones_like(tt)])))
p = [3.0, 1.0, 0.0]
print(f'potential at (3, 1, 0): t = -3: {pot_grow.potential(p, t=-T_total):.1f} (axisymmetric: {pot_axi.potential(p):.1f}); '
      f't = 0: {pot_grow.potential(p, t=0):.1f} (barred: {pot_bar.potential(p):.1f})')
assert abs(pot_grow.potential(p, t=-T_total) / pot_axi.potential(p) - 1) < 1e-10
assert abs(pot_grow.potential(p, t=0) / pot_bar.potential(p) - 1) < 1e-10

The initial disk has AGAMA's quasi-isothermal DF (Chapter 10) with scale length 2.6 kpc and a radial velocity
dispersion of 25 km/s at the Sun, rising inwards with scale length 5.2 kpc — a little colder than the old disk, which
makes resonant features sharper. The DF counts orbits with $J_\phi > 0$ as rotating with the disk, so we flip the sign of
AGAMA's $J_\phi$ for our clockwise disk. We work in the plane ($z = v_z = 0$), as Dehnen (2000) did.

In the bar frame (bar along $x$) the Sun sits at azimuth $180° + 28°$: the bar's near end, at $180°$, is $28°$ ahead of the Sun
in the direction of rotation. The velocity grid covers $-150 < v_R < 150$ and $100 < v_\phi < 320$ km/s in steps of 2 km/s.

In [ ]:
df0 = agama.DistributionFunction(type='QuasiIsothermal', potential=pot_axi, mass=1, Rdisk=2.6, Hdisk=0.3,
                                 sigmar0=25 * np.exp(R0 / 5.2), Rsigmar=5.2)
f0 = lambda xv: df0(af(xv) * np.array([1, 1, -1]))      # f0(J) with J_phi flipped for the clockwise disk

phi_sun = np.pi - bar_angle                               # Sun's azimuth in the bar frame: 180 + 28 deg
e_R = np.array([np.cos(phi_sun), np.sin(phi_sun)])        # outward unit vector at the Sun
e_rot = np.array([np.sin(phi_sun), -np.cos(phi_sun)])     # direction of (clockwise) rotation
vR_g, vphi_g = np.arange(-150, 151, 2.0), np.arange(100, 321, 2.0)
VR, VPHI = np.meshgrid(vR_g, vphi_g, indexing='ij')
ic_grid = np.zeros((VR.size, 6))
ic_grid[:, :2] = R0 * e_R
ic_grid[:, 3:5] = VR.ravel()[:, None] * e_R + VPHI.ravel()[:, None] * e_rot


def velocity_distribution(Omega_b):
    """f(v_R, v_phi) at the Sun today for a bar with pattern speed Omega_b, by backward integration."""
    out = agama.orbit(potential=pot_grow, ic=ic_grid, time=-T_total, timestart=0, Omega=-Omega_b, trajsize=1, verbose=False)
    start = np.array([traj[-1] for traj in out[:, 1]], dtype=float)    # phase-space point at t = -3 (no bar yet)
    return f0(start).reshape(VR.shape)


f_axi = f0(ic_grid).reshape(VR.shape)                     # no bar at all
f_bar = {}
for Ob in (33.3, 39.0, 52.0):
    t0 = time.time(); f_bar[Ob] = velocity_distribution(Ob)
    print(f'Omega_b = {Ob}: {VR.size} orbits in {time.time() - t0:.1f} s')

To see where the resonances are in this plane, we compute the frequencies of every grid point in the axisymmetric
part with the `ActionFinder` and draw the lines where $(\Omega_\phi - \Omega_b)/\Omega_R$ equals $0$ (CR), $-1/4$ (4:1) and $-1/2$ (OLR), the
resonance condition of §13.4 with $l/m = 0$, $-1/4$, $-1/2$ ($\Omega_\phi$ positive for prograde orbits, i.e. AGAMA's $|\Omega_\phi|$ in our frame).
The data are the stars of the `solar_neighbourhood` file within 200 pc, as in Chapter 11.

In [ ]:
sn = load_data('solar_neighbourhood')
col = lambda c: sn[c].to_numpy(float)
K = 4.740470463533348
l, b, pml, pmb = agama.transformCelestialCoords(agama.fromICRStoGalactic, np.radians(col('ra')), np.radians(col('dec')),
                                                col('pmra'), col('pmdec'))
dist = 1 / col('parallax')
xs, ys, zs, vxs, vys, vzs = agama.getGalactocentricFromGalactic(l, b, dist, pml * K, pmb * K, col('radial_velocity'),
                                                                galcen_distance=R0, z_sun=0.0208, galcen_v_sun=(12.9, 245.6, 7.78))
Rs = np.hypot(xs, ys)
vR_d, vphi_d = (xs * vxs + ys * vys) / Rs, -(xs * vys - ys * vxs) / Rs
near = dist < 0.2
H_data, _, _ = np.histogram2d(vR_d[near], vphi_d[near], bins=[np.arange(-151, 152, 2.0), np.arange(99, 322, 2.0)])
print(f'{near.sum():,} stars within 200 pc')

Om_grid = af(ic_grid, frequencies=True)[1]
OmR_g, Omphi_g = Om_grid[:, 0].reshape(VR.shape), np.abs(Om_grid[:, 2]).reshape(VR.shape)

fig, ax = plt.subplots(1, 4, figsize=(21, 5.2), sharey=True)
show = lambda a, f: a.imshow(np.log10(gaussian_filter(f, 0.7) / f.max() + 1e-3).T, origin='lower', aspect='auto',
                             extent=[-151, 151, 99, 321], cmap='Greys', vmin=-2.5, vmax=0)
show(ax[0], H_data); ax[0].set_title(f'Gaia DR3, $d < 200$ pc')
for a, Ob in zip(ax[1:], f_bar):
    show(a, f_bar[Ob]); a.set_title(f'model, $\\Omega_b$ = {Ob} km/s/kpc')
for a, Ob in zip(ax, [None, 33.3, 39.0, 52.0]):
    for Obl, st in ([(33.3, '-'), (52.0, '--')] if Ob is None else [(Ob, '-')]):
        q = (Omphi_g - Obl) / OmR_g
        a.contour(vR_g, vphi_g, q.T, levels=[-0.5, -0.25, 0], colors=['C0', 'C2', 'C3'], linestyles=st, linewidths=1.2)
    a.set_xlabel('$v_R$ [km/s]'); a.set_xlim(-150, 150); a.set_ylim(100, 320)
ax[0].set_ylabel('$v_\\phi$ [km/s]')
plt.tight_layout(); plt.show()

**Figure 13.6.** The distribution of $(v_R, v_\phi)$ at the Sun (log scale). Left: Gaia DR3 stars within 200 pc. Others:
backward-integration models with three pattern speeds. Lines mark resonances computed in the axisymmetric part:
corotation (red), outer 4:1 (green) and OLR (blue); in the data panel, solid lines are for $\Omega_b = 33.3$ and dashed for
$\Omega_b = 52$ km/s/kpc.

Each model has its own structure, and it lies along the resonance lines (Figure 13.6). With the fast bar ($\Omega_b = 52$,
right) the OLR line runs at $v_\phi \approx 205$ km/s and splits the distribution into two modes, just as Dehnen (2000)
described: a main mode and a secondary one at lower $v_\phi$. With the slow bar ($\Omega_b = 33.3$) the corotation line
runs at $v_\phi \approx 185$ km/s, just below the observed Hercules, with only weak structure along it; the 4:1 line crosses
the top of the main mode. The
$\Omega_b = 39$ bar puts the 4:1 resonance through the middle of the main mode and corotation far below it.

We measure two numbers in the data and in the models, from the $v_\phi$ distribution of stars with $0 < v_R < 60$ km/s, where
Hercules is: the position of the **gap** (the deepest minimum between 190 and 225 km/s) and of the **Hercules peak** (the
maximum between 175 and 205 km/s). We compare them with the resonance lines at $v_R = 30$ km/s.

In [ ]:
w_vR = (vR_g > 0) & (vR_g < 60)


def profile(f):
    """f summed over 0 < v_R < 60 km/s, lightly smoothed, as a function of v_phi."""
    return gaussian_filter1d(f[w_vR].sum(axis=0), 1.0)


def gap_and_peak(prof):
    """v_phi of the deepest local minimum in (190, 225) and of the maximum in (175, 205) km/s."""
    w = (vphi_g > 190) & (vphi_g < 225)
    k = argrelmin(prof[w])[0]
    depth = [prof[w][i] / min(prof[w][:i + 1].max(), prof[w][i:].max()) for i in k]   # minimum relative to the lower rim
    gap_v = vphi_g[w][k[np.argmin(depth)]] if len(k) else np.nan
    w2 = (vphi_g > 175) & (vphi_g < 205)
    return gap_v, vphi_g[w2][np.argmax(prof[w2])]


def resonance_line(Ob, level, vR=30):
    """v_phi at which (|Omega_phi| - Omega_b) / Omega_R = level, at the given v_R."""
    q = ((Omphi_g - Ob) / OmR_g)[np.argmin(np.abs(vR_g - vR))]
    return vphi_g[np.argmin(np.abs(q - level))]


prof_data = profile(H_data)
gap_data, herc_data = gap_and_peak(prof_data)
print(f'data: gap at v_phi = {gap_data:.0f} km/s, Hercules peak at {herc_data:.0f} km/s')
print(f'at v_R = 30 km/s: OLR line for Omega_b = 52 at v_phi = {resonance_line(52.0, -0.5):.0f}; '
      f'corotation line for Omega_b = 33.3 at {resonance_line(33.3, 0):.0f} km/s')
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(vphi_g, prof_data / prof_data.max(), 'k', lw=2, label='Gaia DR3')
gaps = {}
for Ob, c in zip(f_bar, ['C3', 'C1', 'C0']):
    pm = profile(f_bar[Ob]); gaps[Ob] = gap_and_peak(pm)[0]
    ax.plot(vphi_g, pm / pm.max(), color=c, label=f'model, $\\Omega_b$ = {Ob}')
    print(f'model Omega_b = {Ob}: gap at v_phi = {gaps[Ob]:.0f} km/s')
ax.axvline(gap_data, color='k', ls=':'); ax.axvline(herc_data, color='k', ls='--')
ax.set_xlabel('$v_\\phi$ [km/s]'); ax.set_ylabel('$f$, stars with $0 < v_R < 60$ km/s (normalised)'); ax.set_xlim(120, 300); ax.legend(fontsize=8)
plt.show()
assert 200 < gap_data < 220 and 185 < herc_data < 205
assert abs(gaps[52.0] - gap_data) < 10                      # the fast-bar model reproduces the gap

# which pattern speeds put a resonance line on the observed features?
Obs = np.arange(25, 60, 0.25)
Ob_OLR = Obs[np.argmin([abs(resonance_line(Ob, -0.5) - gap_data) for Ob in Obs])]
Ob_CR = Obs[np.argmin([abs(resonance_line(Ob, 0) - herc_data) for Ob in Obs])]
print(f'OLR through the gap: Omega_b = {Ob_OLR:.1f};  corotation through the Hercules peak: Omega_b = {Ob_CR:.1f} km/s/kpc')
assert 48 < Ob_OLR < 56 and 29 < Ob_CR < 36

**Figure 13.7.** The distribution of $v_\phi$ for stars with $0 < v_R < 60$ km/s: Gaia data within 200 pc (black) and the three
models; the dotted and dashed lines mark the gap and the Hercules peak in the data.

In the data the gap is at $v_\phi = 212$ km/s and the Hercules peak at 198 km/s (Figure 13.7). Two very different bars explain
them with a resonance: the OLR of a fast bar runs through the gap for $\Omega_b = 51$ km/s/kpc — Dehnen's $1.85\,\Omega_0$ — and
the corotation of a slow bar runs through the Hercules peak for $\Omega_b = 31.5$ km/s/kpc. In the first picture Hercules is the
secondary mode outside the OLR; in the second it is the stars orbiting $L_4$ and $L_5$ (Figure 13.5), seen near their
apocentres. The fast-bar model reproduces the gap ($\Omega_b = 52$: gap at 210 km/s) almost by construction — that is how
Dehnen (2000) measured the pattern speed. Our constant-speed slow bar has resonant structure near the right place but
no clear Hercules at all (Figure 13.7, red).
This is a known shortcoming of steadily rotating models: Chiba, Friske & Schönrich (2021) showed that a bar that
**slows down** (as dynamical friction against the dark halo predicts) sweeps its corotation outwards, traps stars and
drags them along, and so piles up many more stars in Hercules; their decelerating bar reproduces the strength and
the offset of the stream with a present-day $\Omega_b \approx 35$ km/s/kpc. Combined with direct measurements of the bar's length
and pattern speed in the inner Galaxy (Portail et al. 2017; Sanders et al. 2019; Clarke & Gerhard 2022), the current
consensus favours the long, slow bar with corotation at 6–7 kpc.

The lesson goes beyond Hercules: **one resonant feature constrains a resonance, not a model.** The same gap can be a
corotation or an OLR, and only other evidence — the bar's length, or many features at once, such as the 4:1 and OLR
"ridges" of Monari et al. (2019) — breaks the degeneracy. Chapter 14 applies the same ideas to the stellar halo.

## Summary

- AGAMA makes a bar from a built-in triaxial model (Ferrers, Dehnen) or from any density through a `CylSpline` expansion;
  the `rotation` modifier makes it rotate and the `scale` modifier makes it grow.
- Orbits can be integrated in the inertial frame or in the bar frame (`Omega=`); the two agree. In a rotating bar $E$ and
  $L_z$ change, but the Jacobi integral $E_J = E - \Omega_b L_z$ is conserved, to $10^{-7}$ here.
- The effective potential has five Lagrange points; $L_1$, $L_2$, $L_4$, $L_5$ lie near corotation (6.1 kpc for $\Omega_b = 37.5$ km/s/kpc).
- Resonances $m(\Omega_\phi - \Omega_b) = l\Omega_R$ organise the orbits: x1 orbits support the bar, stars are trapped around $L_4$/$L_5$ at
  corotation, and orbits flip their alignment across the OLR.
- Backward integration predicts $f(\mathbf v)$ at the Sun without noise. Hercules can be explained by the OLR of a fast bar
  ($\Omega_b \approx 51$ km/s/kpc) or by the corotation of a slow bar ($\Omega_b \approx 31$–35, with a slowing bar); the bar's length favours the slow bar.

| task | AGAMA |
|---|---|
| a bar from a density | `agama.Potential(type='CylSpline', density=dens, mmax=8, ...)`; `mmax=0` for its axisymmetric part |
| a rotating bar | `agama.Potential(potential=pot, rotation=[[0, a0], [1, a0 + Omega]])` |
| a growing bar | `agama.Potential(potential=pot, scale=np.column_stack([t, A(t), 1]))` |
| orbits in the bar frame | `agama.orbit(potential=pot, ic=ic, time=T, Omega=Omega, verbose=False)` (velocities are inertial) |
| backward integration | `agama.orbit(..., time=-T, timestart=0, verbose=False)` |
| orbital frequencies | `agama.ActionFinder(pot)(xv, frequencies=True)` → $(J_r, J_z, J_\phi)$, $(\Omega_R, \Omega_z, \Omega_\phi)$ |

## Exercises

1. **Bar angle.** Recompute Figure 13.6 for bar angles of 20° and 35°. How much do the features move? Dehnen (2000) found
   the bimodality for bar angles between 10° and 70°.
2. **A slowing bar.** Make the bar decelerate: replace the constant rotation by a table of angles
   $\phi_b(t) = -\int \Omega_b(t)\,dt$ with $\Omega_b$ falling from 50 to 33 km/s/kpc over the last 2 Gyr (the rate of Chiba et al. 2021 is
   about 4.5 km/s/kpc/Gyr). Integrate in the inertial frame with the `rotation` modifier, and rotate the Sun's position
   with the bar. Does Hercules become stronger?
3. **The Ferrers bar.** Replace the Portail/Sormani bar by the Ferrers bar of §13.1 (same mass and length). Where are the
   Lagrange points, and how does Figure 13.6 change?
4. **Elsewhere in the disk.** Predict $f(\mathbf v)$ at $R = 7$ and 9 kpc at the Sun's azimuth. Pérez-Villegas et al. (2017) predict
   that Hercules is stronger inside the Solar circle; check this with the data of Chapter 11 by splitting the stars by $R$.
5. **Chaos.** Integrate orbits started at rest at points between $L_4$ and $L_1$ with `lyapunov=True` in `agama.orbit`. Which
   ones are chaotic?

## References

- R. Chiba, J. K. S. Friske & R. Schönrich (2021), "Resonance sweeping by a decelerating Galactic bar", MNRAS 500, 4710:
  https://ui.adsabs.harvard.edu/abs/2021MNRAS.500.4710C
- J. P. Clarke & O. Gerhard (2022), "The pattern speed of the Milky Way bar/bulge from VIRAC and Gaia", MNRAS 512, 2171:
  https://ui.adsabs.harvard.edu/abs/2022MNRAS.512.2171C
- G. Contopoulos (1980), "How far do bars extend", A&A 81, 198: https://ui.adsabs.harvard.edu/abs/1980A&A....81..198C
- G. Contopoulos & P. Grosbøl (1989), "Orbits in barred galaxies", A&ARv 1, 261: https://ui.adsabs.harvard.edu/abs/1989A&ARv...1..261C
- W. Dehnen (2000), "The effect of the outer Lindblad resonance of the Galactic bar on the local stellar velocity
  distribution", AJ 119, 800: https://ui.adsabs.harvard.edu/abs/2000AJ....119..800D
- G. H. Hunter et al. (2024), A&A 692, A216: https://ui.adsabs.harvard.edu/abs/2024A&A...692A.216H
- A. J. Kalnajs (1991), "Pattern speeds of density waves", in Dynamics of Disc Galaxies, 323:
  https://ui.adsabs.harvard.edu/abs/1991dodg.conf..323K
- G. Monari et al. (2019), "Signatures of the resonances of a large Galactic bar in local velocity space", A&A 626, A41:
  https://ui.adsabs.harvard.edu/abs/2019A&A...626A..41M
- A. Pérez-Villegas et al. (2017), "Revisiting the tale of Hercules", ApJL 840, L2: https://ui.adsabs.harvard.edu/abs/2017ApJ...840L...2P
- D. Pfenniger (1984), "The 3D dynamics of barred galaxies", A&A 134, 373: https://ui.adsabs.harvard.edu/abs/1984A&A...134..373P
- M. Portail et al. (2017), "Dynamical modelling of the galactic bulge and bar", MNRAS 465, 1621:
  https://ui.adsabs.harvard.edu/abs/2017MNRAS.465.1621P
- J. L. Sanders, L. Smith & N. W. Evans (2019), "The pattern speed of the Milky Way bar from transverse velocities",
  MNRAS 488, 4552: https://ui.adsabs.harvard.edu/abs/2019MNRAS.488.4552S
- M. C. Sormani et al. (2022), "The stellar mass distribution of the Milky Way's bar: an analytical model", MNRAS 514, L1:
  https://ui.adsabs.harvard.edu/abs/2022MNRAS.514L...1S
- P. Vauterin & H. Dejonghe (1997), MNRAS 286, 812: https://ui.adsabs.harvard.edu/abs/1997MNRAS.286..812V
- C. Wegg, O. Gerhard & M. Portail (2015), "The structure of the Milky Way's bar outside the bulge", MNRAS 450, 4050:
  https://ui.adsabs.harvard.edu/abs/2015MNRAS.450.4050W
- Data: V. Belokurov (2026), "Galactic Dynamics Tutorial data", Zenodo, https://doi.org/10.5281/zenodo.23646482 — please also cite
  the sources listed in its README (Gaia DR3, Katz et al. 2023).